# PROJEKT
## ZASTOSOWANIE JĘZYKA PYTHON W DATA SCIENCE
ALEKSANDER GLEGOŁA  
73068  
VISTULA  

# 1. Import bibliotek

In [1]:
# instalacja bibliotek gdyby nie były zainstalowane
# !pip install transformers torch scikit-learn pandas

In [2]:
# import potrzebnych bibliotek
import pandas as pd
import numpy as np
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity
import json
import random

# 2. Import danych

In [3]:
# wczytywanie danych
# źródło: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata?resource=download&select=tmdb_5000_movies.csv
# plik został umieszczony na githubie
url = 'https://raw.githubusercontent.com/AleksanderGlegola/DataScienceProject/refs/heads/main/tmdb_5000_movies.csv'
df = pd.read_csv(url)
df.head() # sprawdzenie czy dane się wczytały

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


# 3. Analiza

In [4]:
# analiza
shape = df.shape                    # wymiary
column_names = df.columns           # nazwy kolumn
types = df.dtypes                   # typy danych
number = df.count().sort_values()   # liczba rekordów
missing_counts = df.isnull().sum()  # liczba pustych komórek
print(f"\
Wymiary: \n{shape}\n\n\
Kolumny: \n{column_names}\n\n\
Typy: \n{types}\n\n\
Uzupełnione pola: \n{number}\n\n\
Liczba brakujących wartości: \n{missing_counts}")

Wymiary: 
(4803, 20)

Kolumny: 
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

Typy: 
budget                    int64
genres                   object
homepage                 object
id                        int64
keywords                 object
original_language        object
original_title           object
overview                 object
popularity              float64
production_companies     object
production_countries     object
release_date             object
revenue                   int64
runtime                 float64
spoken_languages         object
status                   object
tagline                  object
title                    object
vote_average            float64
vote_count   

In [5]:
df[['title', 'genres', 'overview']][0:10] # kolumny uzyte w projekcie

,title,genres,overview
0,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","Captain Barbossa, long believed to be dead, ha..."
2,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",Following the death of District Attorney Harve...
4,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","John Carter is a war-weary, former military ca..."
5,Spider-Man 3,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""na...",The seemingly invincible Spider-Man goes up ag...
6,Tangled,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 10751...",When the kingdom's most wanted-and most charmi...
7,Avengers: Age of Ultron,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",When Tony Stark tries to jumpstart a dormant p...
8,Harry Potter and the Half-Blood Prince,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","As Harry begins his sixth year at Hogwarts, he..."
9,Batman v Superman: Dawn of Justice,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",Fearing the actions of a god-like Super Hero l...


# 4. Porządkowanie danych

In [6]:
# porządkowanie
# wypełnienie pustych wartości
df['overview'] = df['overview'].fillna('')

# zamiana jsona gatunków na string
def extract_genres(genre_str):
    try:
        genres = json.loads(genre_str)
        return " ".join([g['name'] for g in genres])
    except:
        return ""

df['genres_clean'] = df['genres'].apply(extract_genres) # utworzenie nowej kolumny z wypisanymi gatunkami

In [7]:
df['genres_clean'][0:10]

,genres_clean
0,Action Adventure Fantasy Science Fiction
1,Adventure Fantasy Action
2,Action Adventure Crime
3,Action Crime Drama Thriller
4,Action Adventure Science Fiction
5,Fantasy Action Adventure
6,Animation Family
7,Action Adventure Science Fiction
8,Adventure Fantasy Family
9,Action Adventure Fantasy


In [8]:
# połączenie gatunków i opis fabuły
df['combined_text'] = df['genres_clean'] + " " + df['overview']

In [9]:
df[['combined_text', 'title']][0:10]

,combined_text,title
0,Action Adventure Fantasy Science Fiction In th...,Avatar
1,"Adventure Fantasy Action Captain Barbossa, lon...",Pirates of the Caribbean: At World's End
2,Action Adventure Crime A cryptic message from ...,Spectre
3,Action Crime Drama Thriller Following the deat...,The Dark Knight Rises
4,Action Adventure Science Fiction John Carter i...,John Carter
5,Fantasy Action Adventure The seemingly invinci...,Spider-Man 3
6,Animation Family When the kingdom's most wante...,Tangled
7,Action Adventure Science Fiction When Tony Sta...,Avengers: Age of Ultron
8,Adventure Fantasy Family As Harry begins his s...,Harry Potter and the Half-Blood Prince
9,Action Adventure Fantasy Fearing the actions o...,Batman v Superman: Dawn of Justice


# 5. Ładowanie modelu

In [10]:
# ładowanie modelu z huggingface transformers z użyciem pipeline
extractor = pipeline("feature-extraction", model="sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [11]:
print(f'{type(extractor.model)} {extractor.model.config.name_or_path}') # sprawdzenie załadowanego modelu
# class 'transformers.models.bert.modeling_bert.BertModel'> sentence-transformers/all-MiniLM-L6-v2

<class 'transformers.models.bert.modeling_bert.BertModel'> sentence-transformers/all-MiniLM-L6-v2


In [12]:
# test
test_text = ["A group of scientists travel through time."]
test_vector = extractor(test_text)

In [13]:
test_vector # podgląd testowego wektora

[[[[-0.12500961124897003,
    0.04684446007013321,
    0.28042536973953247,
    0.4146181344985962,
    0.08147222548723221,
    -0.10760804265737534,
    -0.19736722111701965,
    -0.039009612053632736,
    0.14230209589004517,
    -0.007588270585983992,
    0.03306703642010689,
    -0.1422772854566574,
    -0.5679792165756226,
    0.15602126717567444,
    -0.3776222765445709,
    -0.34333857893943787,
    -0.23028722405433655,
    -0.02858954668045044,
    0.22711153328418732,
    -0.17114491760730743,
    -0.3428175747394562,
    0.07367226481437683,
    0.1985018402338028,
    0.23038020730018616,
    -0.11654341965913773,
    0.17725713551044464,
    -0.2533400058746338,
    -0.24485383927822113,
    0.17419393360614777,
    -0.412802129983902,
    -0.3961118161678314,
    0.05478297173976898,
    -0.496651291847229,
    -0.012530950829386711,
    -0.2831517457962036,
    0.18048444390296936,
    -0.026384912431240082,
    -0.05455079302191734,
    0.26275065541267395,
    0.14970

In [14]:
print(f"Wymiar wektora: {np.array(test_vector).shape}")
# (1, 1, 10, 384) - 1 przekazane zdanie, 1 warstwa, 10 tokenów, 384 wymiarów

Wymiar wektora: (1, 1, 10, 384)


# 6. Tokenizowanie

In [15]:
# funkcja kodująca
def get_sentence_embedding(text_list):
    features = extractor(text_list, truncation=True) # wektory

    embeddings = []

    for movie_tokens in features:
      movie_vector = np.mean(movie_tokens, axis=1)
      embeddings.append(movie_vector)

    return np.array(embeddings)

In [16]:
%%time
# %%time - pomiar czasu

# kodowanie opisów na wektory (40s)
# co 500 filmów informacja o trwającym przetwarzaniu
# GPU:
# CPU times: user 38.9 s, sys: 1.46 s, total: 40.4 s
# Wall time: 40.6 s


batch_size = 500
all_embeddings = []

for i in range(0, len(df), batch_size):
    batch_texts = df['combined_text'].iloc[i:i+batch_size].tolist()
    batch_emb = get_sentence_embedding(batch_texts)
    all_embeddings.append(batch_emb)
    print(f"{i//batch_size + 1}/{len(df)//batch_size + 1}")

# tabela z zakodowanymi filmami
movie_embeddings = np.vstack(all_embeddings)
movie_embeddings = movie_embeddings.squeeze(axis=1)


1/10
2/10
3/10
4/10
5/10
6/10
7/10
8/10


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


9/10
10/10
CPU times: user 39.3 s, sys: 1.45 s, total: 40.8 s
Wall time: 41 s


In [17]:
# zapisanie tokenów:
# np.save('movie_embeddings.npy', movie_embeddings)
# link do githuba:
# https://github.com/AleksanderGlegola/DataScienceProject/raw/refs/heads/main/movie_embeddings.npy

In [18]:
print(movie_embeddings.shape) # rozmiar
movie_embeddings[0:10]

(4803, 384)


array([[ 0.01046226,  0.10285392,  0.07232867, ..., -0.06489653,
        -0.11245102, -0.01052745],
       [-0.10193018, -0.15163373, -0.24282401, ..., -0.40726754,
        -0.2027139 , -0.04137015],
       [-0.18795561,  0.10316822, -0.06672248, ..., -0.12702616,
        -0.09362258, -0.14538914],
       ...,
       [-0.04654187, -0.02167249,  0.01644424, ..., -0.23778178,
        -0.26724738, -0.05181176],
       [-0.1056946 ,  0.07714529,  0.04477665, ...,  0.06030496,
        -0.07040969,  0.06834619],
       [ 0.08123052,  0.03557537, -0.15810834, ..., -0.11205386,
        -0.06775663,  0.11388425]])

# 6. Rekomendowanie

In [19]:
def recommend_movies(user_plot, top_n=5, list_only=False):
    # wektoryzowanie podanego opisu
    user_embedding = get_sentence_embedding([user_plot])
    user_embedding = user_embedding.squeeze(axis=1)

    # obliczanie podobieństwa dla wszystkich filmów
    # wartości <-1,1> (1 - idealne dopasowanie)
    similarities = cosine_similarity(user_embedding, movie_embeddings)

    # sortowanie (ostatnie elementy , bo argsort sortuje rosnąco)
    similar_indices = similarities[0].argsort()[-top_n:][::-1]

    # wyniki
    print(f"\nPodany opis: '{user_plot}'\n")
    for i in similar_indices:
        score = similarities[0][i]
        title = df.iloc[i]['title']
        if not list_only:
          if len(df.iloc[i]['overview'])>200:
            overview = df.iloc[i]['overview'][:200] + "..." # ucięcie tekstu
          else:
            overview = df.iloc[i]['overview'][:-1]
          print(f"Tytuł: {title} (Podobieństwo: {score:.2f})")
          print(f"Opis:  {overview}\n")
        else:
          print(f"Tytuł: {title} (Podobieństwo: {score:.2f})")


In [20]:
# test
recommend_movies("A group of scientists travel through time to prevent a disaster.")


Podany opis: 'A group of scientists travel through time to prevent a disaster.'

Tytuł: Left Behind (Podobieństwo: 0.51)
Opis:  A small group of survivors are left behind after millions of people suddenly vanish during the rapture and the world is plunged into chaos and destruction

Tytuł: Mutant World (Podobieństwo: 0.50)
Opis:  A decade after a disastrous meteor impact wipes out most of society, a group of survivalists emerge to find themselves on a twisted version of the old Earth, with a nascent society besieged by vicious...

Tytuł: Project Almanac (Podobieństwo: 0.49)
Opis:  A group of teens discover secret plans of a time machine, and construct one. However, things start to get out of control

Tytuł: Prometheus (Podobieństwo: 0.47)
Opis:  A team of explorers discover a clue to the origins of mankind on Earth, leading them on a journey to the darkest corners of the universe. There, they must fight a terrifying battle to save the future ...

Tytuł: Timecrimes (Podobieństwo: 0.46)

In [21]:
recommend_movies("A group of scientists travel through time to prevent a disaster.", top_n=20, list_only=True)


Podany opis: 'A group of scientists travel through time to prevent a disaster.'

Tytuł: Left Behind (Podobieństwo: 0.51)
Tytuł: Mutant World (Podobieństwo: 0.50)
Tytuł: Project Almanac (Podobieństwo: 0.49)
Tytuł: Prometheus (Podobieństwo: 0.47)
Tytuł: Timecrimes (Podobieństwo: 0.46)
Tytuł: The Divide (Podobieństwo: 0.45)
Tytuł: Fantastic Four (Podobieństwo: 0.44)
Tytuł: About Time (Podobieństwo: 0.42)
Tytuł: In Time (Podobieństwo: 0.42)
Tytuł: Knowing (Podobieństwo: 0.42)
Tytuł: Interstellar (Podobieństwo: 0.42)
Tytuł: Skyline (Podobieństwo: 0.41)
Tytuł: Event Horizon (Podobieństwo: 0.41)
Tytuł: Dawn of the Planet of the Apes (Podobieństwo: 0.40)
Tytuł: Time Changer (Podobieństwo: 0.40)
Tytuł: Tomorrowland (Podobieństwo: 0.40)
Tytuł: Sunshine (Podobieństwo: 0.39)
Tytuł: Mission to Mars (Podobieństwo: 0.39)
Tytuł: Hot Tub Time Machine (Podobieństwo: 0.39)
Tytuł: Lost in Space (Podobieństwo: 0.38)


# 7. Testowanie

In [22]:
# testowanie rekomendowania kolejnych filmów z serii
# Harry Potter
user_plot = 'Harry Potter follows an orphaned boy who discovers he is a wizard at age 11. \
He attends Hogwarts, where he befriends Ron and Hermione. Together, they battle the dark wizard \
Lord Voldemort, who seeks to conquer the magical world and murdered Harry\'s parents when Harry was a baby.'
recommend_movies(user_plot)


Podany opis: 'Harry Potter follows an orphaned boy who discovers he is a wizard at age 11. He attends Hogwarts, where he befriends Ron and Hermione. Together, they battle the dark wizard Lord Voldemort, who seeks to conquer the magical world and murdered Harry's parents when Harry was a baby.'

Tytuł: Harry Potter and the Philosopher's Stone (Podobieństwo: 0.66)
Opis:  Harry Potter has lived under the stairs at his aunt and uncle's house his whole life. But on his 11th birthday, he learns he's a powerful wizard -- with a place waiting for him at the Hogwarts School ...

Tytuł: Harry Potter and the Goblet of Fire (Podobieństwo: 0.64)
Opis:  Harry starts his fourth year at Hogwarts, competes in the treacherous Triwizard Tournament and faces the evil Lord Voldemort. Ron and Hermione help Harry manage the pressure – but Voldemort lurks, awa...

Tytuł: Harry Potter and the Prisoner of Azkaban (Podobieństwo: 0.59)
Opis:  Harry, Ron and Hermione return to Hogwarts for another magic-filled ye

In [23]:
recommend_movies(user_plot, top_n=20, list_only=True)


Podany opis: 'Harry Potter follows an orphaned boy who discovers he is a wizard at age 11. He attends Hogwarts, where he befriends Ron and Hermione. Together, they battle the dark wizard Lord Voldemort, who seeks to conquer the magical world and murdered Harry's parents when Harry was a baby.'

Tytuł: Harry Potter and the Philosopher's Stone (Podobieństwo: 0.66)
Tytuł: Harry Potter and the Goblet of Fire (Podobieństwo: 0.64)
Tytuł: Harry Potter and the Prisoner of Azkaban (Podobieństwo: 0.59)
Tytuł: Harry Potter and the Chamber of Secrets (Podobieństwo: 0.59)
Tytuł: Harry Potter and the Half-Blood Prince (Podobieństwo: 0.55)
Tytuł: Harry Potter and the Order of the Phoenix (Podobieństwo: 0.53)
Tytuł: Pan's Labyrinth (Podobieństwo: 0.45)
Tytuł: Disturbia (Podobieństwo: 0.44)
Tytuł: Dude Where's My Dog? (Podobieństwo: 0.43)
Tytuł: The Exorcist (Podobieństwo: 0.43)
Tytuł: Bogus (Podobieństwo: 0.42)
Tytuł: Something's Gotta Give (Podobieństwo: 0.41)
Tytuł: St. Vincent (Podobieństwo: 0.41)

In [24]:
# Lord of the rings
user_plot = 'The Hobbit Frodo Baggins inherits the One Ring, an artifact of \
immense power created by the Dark Lord Sauron. To save Middle-earth, Frodo and \
eight companions—the Fellowship—embark on a perilous quest to Mount Doom in \
Mordor, the only place where the Ring can be destroyed'
recommend_movies(user_plot)


Podany opis: 'The Hobbit Frodo Baggins inherits the One Ring, an artifact of immense power created by the Dark Lord Sauron. To save Middle-earth, Frodo and eight companions—the Fellowship—embark on a perilous quest to Mount Doom in Mordor, the only place where the Ring can be destroyed'

Tytuł: The Lord of the Rings: The Fellowship of the Ring (Podobieństwo: 0.88)
Opis:  Young hobbit Frodo Baggins, after inheriting a mysterious ring from his uncle Bilbo, must leave his home in order to keep it from falling into the hands of its evil creator. Along the way, a fellowshi...

Tytuł: The Lord of the Rings: The Return of the King (Podobieństwo: 0.59)
Opis:  Aragorn is revealed as the heir to the ancient kings as he, Gandalf and the other members of the broken fellowship struggle to save Gondor from Sauron's forces. Meanwhile, Frodo and Sam bring the ring...

Tytuł: The Lord of the Rings: The Two Towers (Podobieństwo: 0.56)
Opis:  Frodo and Sam are trekking to Mordor to destroy the One Ring 

In [25]:
# James Bond
user_plot = 'James Bond travels to Jamaica to investigate the murder of a \
British agent. He discovers that a megalomaniacal scientist named Dr. No, \
a member of the criminal organization SPECTRE, is using a secret atomic reactor \
on the island of Crab Key to sabotage American space rocket launches'
recommend_movies(user_plot)


Podany opis: 'James Bond travels to Jamaica to investigate the murder of a British agent. He discovers that a megalomaniacal scientist named Dr. No, a member of the criminal organization SPECTRE, is using a secret atomic reactor on the island of Crab Key to sabotage American space rocket launches'

Tytuł: Dr. No (Podobieństwo: 0.74)
Opis:  In the film that launched the James Bond saga, Agent 007 battles mysterious Dr. No, a scientific genius bent on destroying the U.S. space program. As the countdown to disaster begins, Bond must go to ...

Tytuł: Octopussy (Podobieństwo: 0.67)
Opis:  James Bond is sent to investigate after a fellow “00” agent is found dead with a priceless Farberge egg. James bond follows the mystery and uncovers a smuggling scandal and a Russian General who wants...

Tytuł: Live and Let Die (Podobieństwo: 0.64)
Opis:  James Bond must investigate a mysterious murder case of a British agent in New Orleans. Soon he finds himself up against a gangster boss named Mr. Big

In [26]:
# filmy Marvela
# Iron Man
user_plot = 'Billionaire industrialist Tony Stark is kidnapped by terrorists and \
forced to build a weapon. Instead, he creates a high-tech armored suit to escape. \
Returning home, Stark shuts down his weapons manufacturing and perfects the armor \
to fight global corruption, ultimately taking down his greedy business partner'
recommend_movies(user_plot)


Podany opis: 'Billionaire industrialist Tony Stark is kidnapped by terrorists and forced to build a weapon. Instead, he creates a high-tech armored suit to escape. Returning home, Stark shuts down his weapons manufacturing and perfects the armor to fight global corruption, ultimately taking down his greedy business partner'

Tytuł: Iron Man (Podobieństwo: 0.72)
Opis:  After being held captive in an Afghan cave, billionaire engineer Tony Stark creates a unique weaponized suit of armor to fight evil

Tytuł: Iron Man 2 (Podobieństwo: 0.64)
Opis:  With the world now aware of his dual life as the armored superhero Iron Man, billionaire inventor Tony Stark faces pressure from the government, the press and the public to share his technology with t...

Tytuł: Lords of London (Podobieństwo: 0.54)
Opis:  Tony is a notorious gangster with a big problem. He has woken up in an abandoned farmhouse, with blood on his shirt, and no memory of how he got there. He stumbles into a small town and discove

# 8. Generowanie trójek

In [27]:
# założenie: filmy z tego samego gatunku są bardziej podobne niż filmy z innego gatunku
# tworzę trójki:
# - opis filmu (kotwica)
# - losowy film z tymi samymi gatunkami
# - losowy film z innymi gatunkami

In [28]:
# przykladowe filmy o podobnych gatunkach
df[['title', 'genres_clean', 'combined_text']][:200].sort_values(by='genres_clean')[:20]

,title,genres_clean,combined_text
44,Furious 7,Action,Action Deckard Shaw seeks revenge against Domi...
162,Stealth,Action,Action Deeply ensconced in a top-secret milita...
62,The Legend of Tarzan,Action Adventure,"Action Adventure Tarzan, having acclimated to ..."
21,Robin Hood,Action Adventure,Action Adventure When soldier Robin happens up...
152,Kung Fu Panda 3,Action Adventure Animation Comedy Family,Action Adventure Animation Comedy Family Conti...
187,Puss in Boots,Action Adventure Animation Family Fantasy,Action Adventure Animation Family Fantasy Long...
164,Lethal Weapon 4,Action Adventure Comedy Crime Thriller,Action Adventure Comedy Crime Thriller In the ...
167,Sahara,Action Adventure Comedy Drama Mystery,Action Adventure Comedy Drama Mystery Scouring...
150,Men in Black II,Action Adventure Comedy Science Fiction,Action Adventure Comedy Science Fiction Kay an...
70,Wild Wild West,Action Adventure Comedy Science Fiction Western,Action Adventure Comedy Science Fiction Wester...


In [29]:
# ziarno losowości
random.seed(42)

In [30]:
# kolumna z listą gatunków
df['genre_list'] = df['genres_clean'].apply(lambda x: x.split() if pd.notna(x) else [])

In [31]:
triplets = []

In [32]:
# funkcja generująca trójki dla każdego filmu
def generate_triplets():
  for idx, row in df.iterrows():
      anchor_text = row['combined_text']    # opis fabuły
      anchor_genres = row['genre_list']     # lista gatunków

      if not anchor_genres:                 # pusta lista gatunków
          continue

      # pozytywne przykłady (min. 1 gatunek taki sam)
      pos_mask = df['genre_list'].apply(lambda other_genres: any(g in other_genres for g in anchor_genres))
      pos_mask[idx] = False         # omijamy aktualnie sprawdzany film
      pos_movies = df[pos_mask]     # df z podobnymi filmami

      if pos_movies.empty:          # nie ma podobnych filmów
          continue

      # negatywne przykłady (nie ma nawet 1 wspólnego gatunku)
      neg_mask = df['genre_list'].apply(lambda other_genres: not any(g in other_genres for g in anchor_genres))
      neg_movies = df[neg_mask]

      if neg_movies.empty:        # nie ma nie podobnych filmów
          continue

      # losowa próbka pozytywna i negatywna
      pos_row = pos_movies.sample(n=1).iloc[0]
      neg_row = neg_movies.sample(n=1).iloc[0]

      # dodawanie trójki do listy trójek
      triplets.append({
          'anchor': anchor_text,                  # oryginalny tekst
          'positive': pos_row['combined_text'],   # tekst filmu podobnego
          'negative': neg_row['combined_text']    # tekst filmu nie podobnego
      })


In [33]:
# %%time
# czas generowania trójek (1min)
# CPU times: user 55.1 s, sys: 47.6 ms, total: 55.1 s
# Wall time: 55.4 s
generate_triplets()

CPU times: user 58.9 s, sys: 65.8 ms, total: 59 s
Wall time: 1min 4s


In [34]:
triplets[:2]

[{'anchor': 'Action Adventure Fantasy Science Fiction In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.',
  'positive': 'Action Adventure Fantasy The peaceful realm of Azeroth stands on the brink of war as its civilization faces a fearsome race of invaders: orc warriors fleeing their dying home to colonize another. As a portal opens to connect the two worlds, one army faces destruction and the other faces extinction. From opposing sides, two heroes are set on a collision course that will decide the fate of their family, their people, and their home.',
  'negative': "Comedy Drama Romance Jerry Maguire used to be a typical sports agent: willing to do just about anything he could to get the biggest possible contracts for his clients, plus a nice commission for himself. Then, one day, he suddenly has second thoughts about what he's really doing. When he voices these

# 9. Finetuning

In [35]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

/tmp/ipykernel_3512/1305460641.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


In [36]:
# wczytanie modelu
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [37]:
# przygotowanie danych do trenowania
train_examples = []
for trip in triplets:
    train_examples.append(InputExample(texts=[trip['anchor'], trip['positive'], trip['negative']]))

# ładowanie danych treningowych
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# przypisanie funkcji straty
train_loss = losses.TripletLoss(model=model)

In [38]:
%%timeit
# pomiar czasu:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,               # 3-krotne przejście po danych
    warmup_steps=100,       # pierwsze 100 kroków wolnej nauki
    show_progress_bar=True  # pasek postępu
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,4.158501


Step,Training Loss
500,3.974263


Step,Training Loss
500,3.948892


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
"""
15min

[897/897 03:16, Epoch 3/3]
Step	Training Loss
500	4.044029


[897/897 03:17, Epoch 3/3]
Step	Training Loss
500	3.972924


[897/897 03:20, Epoch 3/3]
Step	Training Loss
500	3.957050


[897/897 03:16, Epoch 3/3]
Step	Training Loss
500	3.934732


[897/897 03:17, Epoch 3/3]
Step	Training Loss
500	3.903447


"""

In [39]:
model.save('finetuned-model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import torch

In [44]:

def recommend_movies_finetuned(user_plot, top_n=5, list_only=False):

    global model

    # generowanie wektorów
    local_embeddings = model.encode(df['combined_text'].tolist(), convert_to_tensor=True).float()

    # kodowanie przekazanej fabuły
    query_embedding = model.encode([user_plot], convert_to_tensor=True).float()

    # obliczanie podobieństwa
    scores = model.similarity(query_embedding, local_embeddings)[0]

    # zmiana na numpy
    scores_np = scores.cpu().numpy()
    top_results = np.argsort(scores_np)[-top_n:][::-1]

    # wyniki
    print(f"\n[DOSTROJONY MODEL] Podany opis: '{user_plot}'\n")
    for idx in top_results:
        score = scores_np[idx]
        title = df.iloc[idx]['title']
        overview = df.iloc[idx]['overview']

        if not list_only:
            if len(overview) > 200:
                short_overview = overview[:200] + "..."
            else:
                short_overview = overview[:-1] if overview else ""
            print(f"Tytuł: {title} (Podobieństwo: {score:.2f})")
            print(f"Opis:  {short_overview}\n")
        else:
            print(f"Tytuł: {title} (Podobieństwo: {score:.2f})")

# 10. Testowanie dostrojonego modelu

In [45]:
# przykładowa fabuła
recommend_movies_finetuned("A group of scientists travel through time to prevent a disaster.")


[DOSTROJONY MODEL] Podany opis: 'A group of scientists travel through time to prevent a disaster.'

Tytuł: Def-Con 4 (Podobieństwo: 0.99)
Opis:  Two men and a woman circle the globe in a satellite armed with a nuclear device. The third world war breaks out, and a few months later the satellite crashes. They survive the crash but one man gets k...

Tytuł: Jumper (Podobieństwo: 0.99)
Opis:  David Rice is a man who knows no boundaries, a Jumper, born with the uncanny ability to teleport instantly to anywhere on Earth. When he discovers others like himself, David is thrust into a dangerous...

Tytuł: Mutant World (Podobieństwo: 0.99)
Opis:  A decade after a disastrous meteor impact wipes out most of society, a group of survivalists emerge to find themselves on a twisted version of the old Earth, with a nascent society besieged by vicious...

Tytuł: Death Calls (Podobieństwo: 0.99)
Opis:  An action-packed love story on the Mexican border featuring oppression, revenge, reincarnation and reb

In [46]:
recommend_movies_finetuned("A group of scientists travel through time to prevent a disaster.", top_n=20, list_only=True)


[DOSTROJONY MODEL] Podany opis: 'A group of scientists travel through time to prevent a disaster.'

Tytuł: Def-Con 4 (Podobieństwo: 0.99)
Tytuł: Jumper (Podobieństwo: 0.99)
Tytuł: Mutant World (Podobieństwo: 0.99)
Tytuł: Death Calls (Podobieństwo: 0.99)
Tytuł: Wolf (Podobieństwo: 0.99)
Tytuł: Transformers (Podobieństwo: 0.99)
Tytuł: The Iron Giant (Podobieństwo: 0.99)
Tytuł: The Angry Birds Movie (Podobieństwo: 0.99)
Tytuł: Show Boat (Podobieństwo: 0.99)
Tytuł: The Lost World: Jurassic Park (Podobieństwo: 0.99)
Tytuł: Anastasia (Podobieństwo: 0.99)
Tytuł: Only the Strong (Podobieństwo: 0.99)
Tytuł: The Good Dinosaur (Podobieństwo: 0.99)
Tytuł: The Book of Mormon Movie, Volume 1: The Journey (Podobieństwo: 0.99)
Tytuł: A LEGO Brickumentary (Podobieństwo: 0.99)
Tytuł: Steamboy (Podobieństwo: 0.99)
Tytuł: Lara Croft: Tomb Raider (Podobieństwo: 0.99)
Tytuł: Lost in Space (Podobieństwo: 0.99)
Tytuł: The Blood of Heroes (Podobieństwo: 0.99)
Tytuł: Shipwrecked (Podobieństwo: 0.99)


In [47]:
# testowanie rekomendowania kolejnych filmów z serii
# Harry Potter
user_plot = 'Harry Potter follows an orphaned boy who discovers he is a wizard at age 11. \
He attends Hogwarts, where he befriends Ron and Hermione. Together, they battle the dark wizard \
Lord Voldemort, who seeks to conquer the magical world and murdered Harry\'s parents when Harry was a baby.'
recommend_movies_finetuned(user_plot)


[DOSTROJONY MODEL] Podany opis: 'Harry Potter follows an orphaned boy who discovers he is a wizard at age 11. He attends Hogwarts, where he befriends Ron and Hermione. Together, they battle the dark wizard Lord Voldemort, who seeks to conquer the magical world and murdered Harry's parents when Harry was a baby.'

Tytuł: Transformers (Podobieństwo: 1.00)
Opis:  Young teenager, Sam Witwicky becomes involved in the ancient struggle between two extraterrestrial factions of transforming robots – the heroic Autobots and the evil Decepticons. Sam holds the clue to...

Tytuł: Jumper (Podobieństwo: 1.00)
Opis:  David Rice is a man who knows no boundaries, a Jumper, born with the uncanny ability to teleport instantly to anywhere on Earth. When he discovers others like himself, David is thrust into a dangerous...

Tytuł: The Lost World: Jurassic Park (Podobieństwo: 1.00)
Opis:  Four years after Jurassic Park's genetically bred dinosaurs ran amok, multimillionaire John Hammond shocks chaos theori

In [48]:
recommend_movies_finetuned(user_plot, top_n=20, list_only=True)


[DOSTROJONY MODEL] Podany opis: 'Harry Potter follows an orphaned boy who discovers he is a wizard at age 11. He attends Hogwarts, where he befriends Ron and Hermione. Together, they battle the dark wizard Lord Voldemort, who seeks to conquer the magical world and murdered Harry's parents when Harry was a baby.'

Tytuł: Transformers (Podobieństwo: 1.00)
Tytuł: Jumper (Podobieństwo: 1.00)
Tytuł: The Lost World: Jurassic Park (Podobieństwo: 1.00)
Tytuł: X-Men: Days of Future Past (Podobieństwo: 1.00)
Tytuł: Def-Con 4 (Podobieństwo: 1.00)
Tytuł: The Blood of Heroes (Podobieństwo: 1.00)
Tytuł: The Angry Birds Movie (Podobieństwo: 1.00)
Tytuł: Lara Croft: Tomb Raider (Podobieństwo: 1.00)
Tytuł: Space Chimps (Podobieństwo: 1.00)
Tytuł: The Good Dinosaur (Podobieństwo: 1.00)
Tytuł: Death Calls (Podobieństwo: 1.00)
Tytuł: The Wolverine (Podobieństwo: 1.00)
Tytuł: Only the Strong (Podobieństwo: 1.00)
Tytuł: Lost in Space (Podobieństwo: 1.00)
Tytuł: Mutant World (Podobieństwo: 1.00)
Tytuł: Stea

In [49]:
# Lord of the rings
user_plot = 'The Hobbit Frodo Baggins inherits the One Ring, an artifact of \
immense power created by the Dark Lord Sauron. To save Middle-earth, Frodo and \
eight companions—the Fellowship—embark on a perilous quest to Mount Doom in \
Mordor, the only place where the Ring can be destroyed'
recommend_movies_finetuned(user_plot)


[DOSTROJONY MODEL] Podany opis: 'The Hobbit Frodo Baggins inherits the One Ring, an artifact of immense power created by the Dark Lord Sauron. To save Middle-earth, Frodo and eight companions—the Fellowship—embark on a perilous quest to Mount Doom in Mordor, the only place where the Ring can be destroyed'

Tytuł: The Lord of the Rings: The Two Towers (Podobieństwo: 1.00)
Opis:  Frodo and Sam are trekking to Mordor to destroy the One Ring of Power while Gimli, Legolas and Aragorn search for the orc-captured Merry and Pippin. All along, nefarious wizard Saruman awaits the Fell...

Tytuł: Harry Potter and the Philosopher's Stone (Podobieństwo: 1.00)
Opis:  Harry Potter has lived under the stairs at his aunt and uncle's house his whole life. But on his 11th birthday, he learns he's a powerful wizard -- with a place waiting for him at the Hogwarts School ...

Tytuł: U.F.O. (Podobieństwo: 1.00)
Opis:  A group of friends awake one morning to find all electricity and power shut off and an imm

In [50]:
# James Bond
user_plot = 'James Bond travels to Jamaica to investigate the murder of a \
British agent. He discovers that a megalomaniacal scientist named Dr. No, \
a member of the criminal organization SPECTRE, is using a secret atomic reactor \
on the island of Crab Key to sabotage American space rocket launches'
recommend_movies_finetuned(user_plot)


[DOSTROJONY MODEL] Podany opis: 'James Bond travels to Jamaica to investigate the murder of a British agent. He discovers that a megalomaniacal scientist named Dr. No, a member of the criminal organization SPECTRE, is using a secret atomic reactor on the island of Crab Key to sabotage American space rocket launches'

Tytuł: Falcon Rising (Podobieństwo: 1.00)
Opis:  Chapman is an ex-marine in Brazil's slums, battling the yakuza outfit who attacked his sister and left her for dead

Tytuł: Star Wars (Podobieństwo: 1.00)
Opis:  Princess Leia is captured and held hostage by the evil Imperial forces in their effort to take over the galactic Empire. Venturesome Luke Skywalker and dashing captain Han Solo team together with the ...

Tytuł: Akira (Podobieństwo: 1.00)
Opis:  Childhood friends Tetsuo and Kaneda are pulled into the post-apocalyptic underworld of Neo-Tokyo and forced to fight for their very survival. Kaneda is a bike gang leader, and Tetsuo is a member of a ...

Tytuł: Ultramarine

In [51]:
# filmy Marvela
# Iron Man
user_plot = 'Billionaire industrialist Tony Stark is kidnapped by terrorists and \
forced to build a weapon. Instead, he creates a high-tech armored suit to escape. \
Returning home, Stark shuts down his weapons manufacturing and perfects the armor \
to fight global corruption, ultimately taking down his greedy business partner'
recommend_movies_finetuned(user_plot)


[DOSTROJONY MODEL] Podany opis: 'Billionaire industrialist Tony Stark is kidnapped by terrorists and forced to build a weapon. Instead, he creates a high-tech armored suit to escape. Returning home, Stark shuts down his weapons manufacturing and perfects the armor to fight global corruption, ultimately taking down his greedy business partner'

Tytuł: The Iron Giant (Podobieństwo: 1.00)
Opis:  In the small town of Rockwell, Maine in October 1957, a giant metal machine befriends a nine-year-old boy and ultimately finds its humanity by unselfishly saving people from their own fears and prejud...

Tytuł: The Wolverine (Podobieństwo: 1.00)
Opis:  Wolverine faces his ultimate nemesis - and tests of his physical, emotional, and mortal limits - in a life-changing voyage to modern-day Japan

Tytuł: Earth to Echo (Podobieństwo: 1.00)
Opis:  After a construction project begins digging in their neighborhood, best friends Tuck, Munch and Alex inexplicably begin to receive strange, encoded messages

# 11. Porównanie modeli

In [40]:
model_base_clean = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

model_ft_clean = model

# generowanie wektorów
embeddings_base_clean = model_base_clean.encode(df['combined_text'].tolist(), convert_to_tensor=True).float()

# generowanie wektorów
embeddings_ft_clean = model_ft_clean.encode(df['combined_text'].tolist(), convert_to_tensor=True).float()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


=== KROK 2: GENEROWANIE EMBEDDINGÓW (MAP MAPOWANIA) ===
Generowanie wektorów dla modelu bazowego...
Generowanie wektorów dla modelu dostrojonego...

[SUKCES] Modele i wektory zostały poprawnie odizolowane!


In [55]:
# funkcja porównująca wyniki
def compare_to_dataframe(user_plot, top_n=5):

    # model bazowy
    query_emb_base = model_base_clean.encode([user_plot], convert_to_tensor=True).float()
    scores_base = model_base_clean.similarity(query_emb_base, embeddings_base_clean)[0].cpu().numpy()
    idx_base = np.argsort(scores_base)[-top_n:][::-1]

    # model dostrojony
    query_emb_ft = model_ft_clean.encode([user_plot], convert_to_tensor=True).float()
    scores_ft = model_ft_clean.similarity(query_emb_ft, embeddings_ft_clean)[0].cpu().numpy()
    idx_ft = np.argsort(scores_ft)[-top_n:][::-1]

    # struktura danych
    base_titles, base_scores = [], []
    ft_titles, ft_scores = [], []

    for i in range(top_n):
        # dane bazowe
        b_idx = idx_base[i]
        base_titles.append(df.iloc[b_idx]['title'])
        base_scores.append(round(float(scores_base[b_idx]), 4))

        # dane dostrojone
        ft_idx = idx_ft[i]
        ft_titles.append(df.iloc[ft_idx]['title'])
        ft_scores.append(round(float(scores_ft[ft_idx]), 4))

    # tworzenie df
    comparison_df = pd.DataFrame({
        'film': base_titles,
        'podobieństwo': base_scores,
        'film z dostrojonego modelu': ft_titles,
        'podobieństwo z dostrojonego': ft_scores
    })

    # indeksowanie
    comparison_df.index = np.arange(1, len(comparison_df) + 1)

    return comparison_df

# 12. Przykłady

In [56]:
# test 1: film kryminalny/akcji (Heat)
plot_heat = (
    "A meticulous career thief (Robert De Niro) and a relentless LAPD detective (Al Pacino) "
    "are locked in a deadly cat-and-mouse game across Los Angeles. While tracking a botched heist, "
    "both men realize they share a mutual respect and are fundamentally identical in their obsessive dedication."
)

df_heat = compare_to_dataframe(plot_heat, top_n=5)
display(df_heat)
print("\n")

# test 2: komedia romantyczna
plot_romance = "A romantic comedy about two people who live in New York and accidentally fall in love over Christmas."
df_romance = compare_to_dataframe(plot_romance, top_n=5)

display(df_romance)

,film,podobieństwo,film z dostrojonego modelu,podobieństwo z dostrojonego
1,The Town,0.5842,Jumper,0.9992
2,Kiss Kiss Bang Bang,0.4861,Transformers,0.9992
3,Parker,0.4822,The Iron Giant,0.9992
4,12 Rounds,0.4778,The Good Dinosaur,0.9992
5,American Hustle,0.4754,Lost in Space,0.9989


,film,podobieństwo,film z dostrojonego modelu,podobieństwo z dostrojonego
1,Love Actually,0.7466,Laws of Attraction,0.9999
2,New Year's Eve,0.7003,Intolerable Cruelty,0.9999
3,The Incredibly True Adventure of Two Girls In ...,0.6150,Neighbors,0.9999
4,The Best Man Holiday,0.6113,Waking Ned,0.9999
5,Never Again,0.6069,Bananas,0.9999


# 13. Porównanie rekomendacji

## Harry Potter

In [57]:
# testowanie rekomendowania kolejnych filmów z serii
# Harry Potter
user_plot = 'Harry Potter follows an orphaned boy who discovers he is a wizard at age 11. \
He attends Hogwarts, where he befriends Ron and Hermione. Together, they battle the dark wizard \
Lord Voldemort, who seeks to conquer the magical world and murdered Harry\'s parents when Harry was a baby.'
compare_to_dataframe(user_plot, top_n=30)

,film,podobieństwo,film z dostrojonego modelu,podobieństwo z dostrojonego
1,Harry Potter and the Philosopher's Stone,0.6603,Transformers,0.9964
2,Harry Potter and the Goblet of Fire,0.6389,Jumper,0.9962
3,Harry Potter and the Prisoner of Azkaban,0.5882,The Lost World: Jurassic Park,0.9959
4,Harry Potter and the Chamber of Secrets,0.5850,X-Men: Days of Future Past,0.9959
5,Harry Potter and the Half-Blood Prince,0.5480,Def-Con 4,0.9958
6,Harry Potter and the Order of the Phoenix,0.5262,The Blood of Heroes,0.9958
7,Pan's Labyrinth,0.4522,The Angry Birds Movie,0.9958
8,Disturbia,0.4417,Lara Croft: Tomb Raider,0.9958
9,Dude Where's My Dog?,0.4332,Space Chimps,0.9957
10,The Exorcist,0.4320,The Good Dinosaur,0.9957


## Władca Pierścieni

In [58]:
# Lord of the rings
user_plot = 'The Hobbit Frodo Baggins inherits the One Ring, an artifact of \
immense power created by the Dark Lord Sauron. To save Middle-earth, Frodo and \
eight companions—the Fellowship—embark on a perilous quest to Mount Doom in \
Mordor, the only place where the Ring can be destroyed'
compare_to_dataframe(user_plot, top_n=30)

,film,podobieństwo,film z dostrojonego modelu,podobieństwo z dostrojonego
1,The Lord of the Rings: The Fellowship of the Ring,0.8768,The Lord of the Rings: The Two Towers,0.9987
2,The Lord of the Rings: The Return of the King,0.5941,Harry Potter and the Philosopher's Stone,0.9983
3,The Lord of the Rings: The Two Towers,0.5639,U.F.O.,0.9982
4,The Hobbit: An Unexpected Journey,0.5545,The Bounty Hunter,0.9982
5,The Hobbit: The Desolation of Smaug,0.5088,The Scorpion King,0.9982
6,The Hobbit: The Battle of the Five Armies,0.3849,Excessive Force,0.9981
7,Little Nicky,0.3438,Universal Soldier: The Return,0.9980
8,Warlock: The Armageddon,0.3246,Clash of the Titans,0.9980
9,"The Book of Mormon Movie, Volume 1: The Journey",0.3162,Digimon: The Movie,0.9980
10,Raging Bull,0.3093,The Legend of Tarzan,0.9980


## James Bond

In [59]:
# James Bond
user_plot = 'James Bond travels to Jamaica to investigate the murder of a \
British agent. He discovers that a megalomaniacal scientist named Dr. No, \
a member of the criminal organization SPECTRE, is using a secret atomic reactor \
on the island of Crab Key to sabotage American space rocket launches'
compare_to_dataframe(user_plot, top_n=30)

,film,podobieństwo,film z dostrojonego modelu,podobieństwo z dostrojonego
1,Dr. No,0.7405,Falcon Rising,0.9989
2,Octopussy,0.6674,Star Wars,0.9988
3,Live and Let Die,0.6448,Akira,0.9987
4,The Living Daylights,0.6254,"Ultramarines: A Warhammer 40,000 Movie",0.9986
5,Quantum of Solace,0.5660,Percy Jackson & the Olympians: The Lightning T...,0.9986
6,Never Say Never Again,0.5651,The Lone Ranger,0.9986
7,You Only Live Twice,0.5597,Pirates of the Caribbean: At World's End,0.9985
8,Die Another Day,0.5421,Captain America: The First Avenger,0.9985
9,From Russia with Love,0.5396,The Lord of the Rings: The Fellowship of the Ring,0.9985
10,For Your Eyes Only,0.5310,Six-String Samurai,0.9985


## Filmy Marvela

In [60]:
# filmy Marvela
# Iron Man
user_plot = 'Billionaire industrialist Tony Stark is kidnapped by terrorists and \
forced to build a weapon. Instead, he creates a high-tech armored suit to escape. \
Returning home, Stark shuts down his weapons manufacturing and perfects the armor \
to fight global corruption, ultimately taking down his greedy business partner'
compare_to_dataframe(user_plot, top_n=30)

,film,podobieństwo,film z dostrojonego modelu,podobieństwo z dostrojonego
1,Iron Man,0.7174,The Iron Giant,0.9992
2,Iron Man 2,0.6352,The Wolverine,0.9990
3,Lords of London,0.5435,Earth to Echo,0.9990
4,Iron Man 3,0.5244,The Adventures of Sharkboy and Lavagirl,0.9989
5,Avengers: Age of Ultron,0.5123,The Good Dinosaur,0.9987
6,Steel,0.4972,Yeh Jawaani Hai Deewani,0.9985
7,Scarface,0.4818,Madagascar,0.9985
8,Cradle 2 the Grave,0.4745,Show Boat,0.9985
9,Machete Kills,0.4500,Transformers,0.9985
10,Shinjuku Incident,0.4233,Jumper,0.9983


## Love Actually

In [61]:
user_plot = "Love Actually is a 2003 British holiday romantic comedy \
that weaves together ten distinct stories of love, loss, and longing. \
Set in London during the frantic weeks leading up to Christmas, \
the film explores the complexities of relationships through a massive, \
interconnected cast of characters."
compare_to_dataframe(user_plot, top_n=30)

,film,podobieństwo,film z dostrojonego modelu,podobieństwo z dostrojonego
1,Love Actually,0.7088,Cars,0.9994
2,About Last Night,0.5827,Wreck-It Ralph,0.9994
3,The Incredibly True Adventure of Two Girls In ...,0.5819,All About the Benjamins,0.9994
4,"Lovely, Still",0.5817,The Men Who Stare at Goats,0.9993
5,The Lovers,0.5701,Little Shop of Horrors,0.9993
6,New Year's Eve,0.5684,3 Ninjas Kick Back,0.9993
7,Midnight in Paris,0.5578,Ice Age,0.9993
8,Before Sunrise,0.5566,Mortdecai,0.9993
9,Housefull,0.5546,Inspector Gadget,0.9993
10,The Broken Hearts Club: A Romantic Comedy,0.5522,Ravenous,0.9993


Wybrana metoda dostrajania okazała się przeciwskuteczna. Rekomendacje proponowane przez pierwszy model były dużo bardziej podobne do przedstawionego filmu.